# CS 5326 PA1 — Kaggle orchestration (2× T4, fp16 mixed precision)

This notebook only **orchestrates**; all logic lives in the repository's `src/` CLIs:

| Stage | Command |
|---|---|
| unit tests | `pytest` |
| fp16 dtype audit, fp32↔fp16 parity, throughput benchmark | `src/precision_check.py` |
| single-batch overfit sanity check | `src/train.py --overfit-single-batch` |
| short ablation runs (one per GPU, in parallel) | `src/train.py` |
| final 10,000-update run on both GPUs | `torchrun --nproc_per_node=2 src/train.py` |
| figures, standardized eval, decoding study, export | `src/plot_metrics.py`, `src/evaluate.py`, `src/generate.py`, `src/export_model.py` |

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **on**. For long runs use
*Save Version → Save & Run All (Commit)* so training continues with the browser closed.
Everything under `/kaggle/working` is kept as the version's output.

**Resuming after a session ends:** add the previous version's output as an input
(*Add Input → Your Work → this notebook*), set `RESUME_FROM` below to the checkpoint
inside `/kaggle/input/...`, and run again. `--time-limit-hours` makes the run
checkpoint and exit cleanly before Kaggle's 12-hour limit.

## 1. Configuration

In [ ]:
import os, subprocess, sys, shutil, json, time
from pathlib import Path

# --- where the code comes from -------------------------------------------------
REPO_URL = "https://github.com/<your-username>/cs5326-pa1.git"   # your fork / private repo
BRANCH = "main"
USE_GITHUB_TOKEN_SECRET = True    # Kaggle secret named GITHUB_TOKEN (needed for private repos)
CODE_DATASET_DIR = None           # alternatively: "/kaggle/input/<dataset-with-repo-snapshot>"

WORK = Path("/kaggle/working")
REPO = WORK / "cs5326-pa1"
DATA = Path("/tmp/tinystories")          # large token streams: kept OUT of /kaggle/working outputs
RUNS = WORK / "runs"

# --- which stages to run ---------------------------------------------------------
RUN_TESTS = True
RUN_PRECISION_CHECK = True        # also writes report_assets/precision_benchmark.json
RUN_OVERFIT_CHECK = False
RUN_ABLATIONS = False             # short tuning runs, two at a time (one per GPU)
RUN_FINAL = True                  # the 10,000-update run on both GPUs
RUN_POSTPROCESS = True            # plots, standardized eval, decoding study, final_model.pt

FINAL_RUN_NAME = "final"
RESUME_FROM = None                # e.g. "/kaggle/input/<prev-version>/runs/final/checkpoint.pt"
TIME_LIMIT_HOURS = 11.0           # checkpoint + exit before Kaggle's 12 h session cap

# Final-run hyperparameters: Section 7.1 starting point with the peak LR chosen by the
# ablations (1e-3 beat 6e-4 and 3e-4 at every evaluation, with no instability);
# --batch-size is the microbatch summed over both GPUs: 64 x 4 = 256 sequences/update,
# i.e. 32 sequences per T4 per microbatch). --compile fuses the hand-written elementwise
# ops; on a T4 it lifts fp16 throughput from ~41k to ~99k tokens/s (precision benchmark).
FINAL_ARGS = [
    "--num-steps", "10000", "--cosine-steps", "9999",
    "--batch-size", "64", "--grad-accum", "4", "--sequence-length", "256",
    "--lr-max", "1e-3", "--lr-min", "1e-4", "--warmup-steps", "200",
    "--beta1", "0.9", "--beta2", "0.95", "--adam-eps", "1e-8",
    "--weight-decay", "0.1", "--max-grad-norm", "1.0",
    "--eval-interval", "250", "--log-interval", "25", "--checkpoint-interval", "250",
    "--num-val-batches", "20", "--precision", "fp16", "--compile",
]

## 2. GPU check

In [ ]:
!nvidia-smi
driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.split()
print("driver:", driver)

## 3. Get the code

In [ ]:
if CODE_DATASET_DIR:
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(CODE_DATASET_DIR, REPO)
elif (REPO / ".git").exists():
    !git -C {REPO} pull --ff-only
else:
    url = REPO_URL
    if USE_GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    # The token never gets printed; strip it from the stored remote after cloning.
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO)
!git log --oneline -3

## 4. Environment

`uv sync --frozen` installs the locked environment. The locked torch wheel is built for
**CUDA 13**, which needs NVIDIA driver ≥ 580. If the locked torch cannot see the GPUs on
this Kaggle image, the notebook falls back to Kaggle's preinstalled torch and adds only the
missing pure-Python packages. The code is identical either way.

In [ ]:
!pip install -q uv
!uv sync --frozen -q

probe = subprocess.run(
    ["uv", "run", "python", "-c", "import torch; assert torch.cuda.is_available() and torch.cuda.device_count() >= 1; print(torch.__version__, torch.version.cuda, torch.cuda.device_count())"],
    capture_output=True, text=True)
print(probe.stdout, probe.stderr[-2000:])
if probe.returncode == 0:
    PY, TORCHRUN = ["uv", "run", "python"], ["uv", "run", "torchrun"]
    print("using the locked uv environment")
else:
    !pip install -q "einops>=0.8,<1" "jaxtyping>=0.3,<1" "wrapt>=1.17" "pytest>=8.3,<9"
    PY, TORCHRUN = [sys.executable], [sys.executable, "-m", "torch.distributed.run"]
    print("FALLBACK: using Kaggle's preinstalled torch")
subprocess.run(PY + ["-c", "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"], check=True)

In [ ]:
def run(args, env=None, log_path=None):
    """Run a command, stream its output, optionally tee it to a log file; raise on failure."""
    print("$", " ".join(map(str, args)), flush=True)
    full_env = {**os.environ, **(env or {}), "PYTHONUNBUFFERED": "1"}
    log = open(log_path, "a") if log_path else None
    proc = subprocess.Popen(list(map(str, args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env=full_env)
    for line in proc.stdout:
        print(line, end="")
        if log:
            log.write(line); log.flush()
    if log:
        log.close()
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

def run_parallel(jobs):
    """jobs: list of (name, args, env). Runs them concurrently; logs go to RUNS/<name>/train.log."""
    procs = []
    for name, args, env in jobs:
        (RUNS / name).mkdir(parents=True, exist_ok=True)
        log = open(RUNS / name / "train.log", "a")
        print("$", " ".join(map(str, args)), f"  [{env}] -> {RUNS / name / 'train.log'}", flush=True)
        procs.append((name, subprocess.Popen(list(map(str, args)), stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, **env, "PYTHONUNBUFFERED": "1"}), log))
    reported = set()
    while len(reported) < len(procs):
        time.sleep(60)
        for name, p, log in procs:
            if name in reported:
                continue
            lines = (RUNS / name / "train.log").read_text().splitlines()
            print(f"[{name}] {lines[-1] if lines else ''}", flush=True)
            if p.poll() is not None:
                reported.add(name)
                log.close()
                print(f"[{name}] finished with exit code {p.returncode}", flush=True)
    failed = [name for name, p, _ in procs if p.returncode != 0]
    for name in failed:
        print(f"===== {name} FAILED; last lines of {RUNS / name / 'train.log'}:")
        print("\n".join((RUNS / name / "train.log").read_text().splitlines()[-30:]))
    if failed:
        raise RuntimeError(f"runs failed: {failed}")

## 5. Dataset (token streams go to local disk, not to the saved outputs)

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
run(PY + ["-c", f"""
from huggingface_hub import hf_hub_download
for name in ['metadata.json', 'tokenizer/tokenizer.json', 'data/train.bin', 'data/validation.bin']:
    print('downloading', name, flush=True)
    print('  ->', hf_hub_download('alooboii/pa1-tinystories', name, repo_type='dataset', local_dir='{DATA}'), flush=True)
"""], env={"HF_HUB_DISABLE_PROGRESS_BARS": "1"})   # progress bars would print hundreds of lines through run()
TRAIN_BIN, VAL_BIN = DATA / "data/train.bin", DATA / "data/validation.bin"
TOKENIZER = DATA / "tokenizer/tokenizer.json"
print({p.name: p.stat().st_size for p in (TRAIN_BIN, VAL_BIN)})
DATA_ARGS = ["--train-data", TRAIN_BIN, "--val-data", VAL_BIN]

## 6. Unit tests

In [ ]:
if RUN_TESTS:
    run(PY + ["-m", "pytest", "-q"])

## 7. Mixed-precision check and throughput benchmark (single T4)

Audits dtypes under autocast (Linear/LM head fp16; RMSNorm, residual stream, loss,
parameters, gradients fp32), compares one fp32 vs fp16 forward/backward (loss difference,
gradient cosine similarity), and times fp32 vs fp16 steps for several microbatch sizes.
The JSON goes to `report_assets/` as evidence for the report.

In [ ]:
if RUN_PRECISION_CHECK:
    run(PY + ["src/precision_check.py", "--batch-sizes", "16", "32", "--steps", "10", "--compile",
              "--out", "report_assets/precision_benchmark.json"], env={"CUDA_VISIBLE_DEVICES": "0"})

## 8. Overfit one fixed minibatch (Section 5.3 sanity check)

In [ ]:
if RUN_OVERFIT_CHECK:
    run(PY + ["src/train.py", *DATA_ARGS, "--run-dir", RUNS / "overfit", "--resume", "none",
              "--overfit-single-batch", "--num-steps", "150", "--batch-size", "16", "--grad-accum", "1",
              "--lr-max", "1e-3", "--lr-min", "1e-4", "--warmup-steps", "10", "--cosine-steps", "150",
              "--weight-decay", "0.0", "--eval-interval", "50", "--log-interval", "10",
              "--checkpoint-interval", "1000", "--num-val-batches", "2"],
        env={"CUDA_VISIBLE_DEVICES": "0"})

## 9. Controlled short-run ablations (Section 7.2) — two runs at a time, one per GPU

Every run shares seeds, the evaluation procedure, and `--cosine-steps 9999` (short runs see
the same LR-schedule prefix as the full run). Edit `ABLATIONS` to build your ablation ladder;
each entry overrides the baseline. Pairs run concurrently on GPU 0 and GPU 1.

In [ ]:
ABLATION_STEPS = "1500"
BASELINE = ["--num-steps", ABLATION_STEPS, "--cosine-steps", "9999", "--batch-size", "32", "--grad-accum", "8",
            "--lr-max", "3e-4", "--lr-min", "3e-5", "--warmup-steps", "200", "--weight-decay", "0.1",
            "--eval-interval", "100", "--log-interval", "25", "--checkpoint-interval", "500",
            "--num-val-batches", "20", "--precision", "fp16", "--compile"]
ABLATIONS = {
    # name: extra args appended to BASELINE (later flags win)
    "base_fp16": [],
    "base_fp32": ["--precision", "fp32"],
    "lr6e-4": ["--lr-max", "6e-4", "--lr-min", "6e-5"],
    "lr1e-3": ["--lr-max", "1e-3", "--lr-min", "1e-4"],
}

if RUN_ABLATIONS:
    names = list(ABLATIONS)
    for i in range(0, len(names), 2):
        jobs = []
        for gpu, name in enumerate(names[i:i + 2]):
            args = PY + ["src/train.py", *DATA_ARGS, "--run-dir", RUNS / name, *BASELINE, *ABLATIONS[name]]
            jobs.append((name, args, {"CUDA_VISIBLE_DEVICES": str(gpu)}))
        run_parallel(jobs)
    run(PY + ["src/plot_metrics.py", *[RUNS / n for n in names], "--labels", *names, "--compare",
              "--out", "report_assets/ablations_val_loss.png"])

## 10. Final run — 10,000 updates on both T4s (fp16 AMP + gradient all-reduce)

Resumes automatically from `runs/<name>/checkpoint.pt` (or from `RESUME_FROM`).

In [ ]:
if RUN_FINAL:
    final_dir = RUNS / FINAL_RUN_NAME
    final_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_FROM and not (final_dir / "checkpoint.pt").exists():
        src_dir = Path(RESUME_FROM).parent
        shutil.copy(RESUME_FROM, final_dir / "checkpoint.pt")
        if (src_dir / "metrics.jsonl").exists():
            shutil.copy(src_dir / "metrics.jsonl", final_dir / "metrics.jsonl")
    run(TORCHRUN + ["--standalone", "--nproc_per_node=2", "src/train.py", *DATA_ARGS,
                    "--run-dir", final_dir, *FINAL_ARGS, "--time-limit-hours", str(TIME_LIMIT_HOURS)],
        log_path=final_dir / "train.log")

## 11. Post-processing: figures, standardized evaluation, decoding study, `final_model.pt`

The standardized evaluation loads the exported **fp16** `final_model.pt` into an fp32 model,
which is exactly what a grader sees. Fill in `REPORT.md` with these results.

In [ ]:
if RUN_POSTPROCESS:
    final_dir = RUNS / FINAL_RUN_NAME
    run(PY + ["src/export_model.py", "--checkpoint", final_dir / "checkpoint.pt", "--out", "final_model.pt"])
    run(PY + ["src/plot_metrics.py", final_dir, "--title", "Final run", "--out", "report_assets/final_run.png"])
    run(PY + ["src/evaluate.py", "--model", "final_model.pt", "--val-data", VAL_BIN,
              "--out", "report_assets/final_eval.json"], env={"CUDA_VISIBLE_DEVICES": "0"})
    run(PY + ["src/generate.py", "--model", "final_model.pt", "--tokenizer", TOKENIZER,
              "--prompt", "Once upon a time", "--prompt", "Tom and his dog went to the park.",
              "--temperatures", "0.3", "0.7", "1.0", "1.3", "--top-ps", "0.5", "0.9", "1.0",
              "--num-samples", "3", "--max-new-tokens", "200",
              "--out", "report_assets/decoding_samples.json"], env={"CUDA_VISIBLE_DEVICES": "0"})

## 12. Package the submission (after `REPORT.md` is complete)

In [ ]:
# bash make_submission.sh needs `uv` on PATH (it runs the tests through `uv run`).
# !bash make_submission.sh && ls -la submission.zip